# 4.0 - Escolha das melhores features engineering do modelo

Dado o conjunto de features enginnering:

- **1.0**: Nan Rows.
- **2.0**: Lags.
- **3.0**: Sin(x),Cos(x).
- **5.0**: Derivadas.
- **6.0**: Coordenadas Solares.
- **7.0**: Features de Clima.
- **8.0**: Models Features(Adicionar os modelos Persistencia ou Modelo).


Metrica: AP por fold do CV temporal (so treino), media dos folds.
Decisão baseado em teste de hipótese e testes estatísticos.
Comparar com a persistência.

In [1]:
%load_ext autoreload
%autoreload 2

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ml_atmospheric_emission.config import PATH_REPORTS, TAGS, CALENDARIO, ESTADO, DIAS_SUSPEITOS, CABECAS_POSSIVEIS, MODELO_ATUAL,PREVISOR_POR_CABECA
from ml_atmospheric_emission.dataset import load_interim, remove_days
from ml_atmospheric_emission.features import create_datetime_columns, create_target, create_state_columns, create_future_targets
from ml_atmospheric_emission.modeling.evaluate import split_holdout, compare_horizons, choose_horizons, choose_forecaster, holdout_ap, save_horizon_report
from ml_atmospheric_emission.plots import plot_horizons

In [7]:
#True: removes the suspicious days (22h of plume in a row, including the night)
#False: sensitivity analysis with all the days
REMOVER_DIAS_SUSPEITOS = True

ds = load_interim()
ds = create_datetime_columns(ds)
ds = create_target(ds)
if REMOVER_DIAS_SUSPEITOS:
    ds = remove_days(ds, DIAS_SUSPEITOS)
ds = create_state_columns(ds)
dx = create_future_targets(ds, 72)   #72 x 5 min = 6 horas no futuro

#The hold out (last 20% of the period) is separated ONCE
dx_treino, dx_holdout = split_holdout(dx)

Treino ate: 2026-10-02 23:15:00 | linhas treino: 11598 | linhas hold out: 2900


In [9]:
dx_treino

,Data,nn_score_smog,int_sd,int_med,306-AIT-004-2,306-AIT-004-5,306FE002-AI001-R,406-AI-004-2,406-AI-004-4,406-AIT-001,...,y_315,y_320,y_325,y_330,y_335,y_340,y_345,y_350,y_355,y_360
0,2026-08-20 14:15:00,1.0,152.000,153.134,71.445,183.992,7.277,43.072,178.567,1.008,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2026-08-20 14:20:00,1.0,152.000,155.969,67.433,186.557,7.291,45.751,178.031,1.036,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-08-20 14:25:00,1.0,152.000,144.478,66.290,181.367,6.485,41.679,174.437,0.919,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-08-20 14:30:00,1.0,152.000,148.253,68.476,174.032,8.987,40.868,176.775,0.555,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2026-08-20 14:35:00,1.0,152.000,149.863,67.240,173.400,7.643,42.162,176.990,0.852,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11593,2026-10-02 22:50:00,0.0,33.000,33.325,62.172,239.748,19.285,127.243,258.270,0.787,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11594,2026-10-02 22:55:00,0.0,30.094,32.254,61.412,237.278,7.195,127.085,268.000,0.485,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11595,2026-10-02 23:00:00,0.0,29.854,31.477,60.070,239.702,5.026,131.606,282.637,0.514,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11596,2026-10-02 23:05:00,0.0,30.000,31.121,60.468,232.897,4.778,132.394,272.372,0.807,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Feature 1.0: Nan Rows

Utilizaremos MICE Linear e MICE não linear e iremos comparar via teste de hipótese, e depois comparar o desempenho com a persistência.

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.ensemble import RandomForestRegressor

def MiceLinear(ds_treino,ds_teste):
    ds = ds_treino.copy()
    dst = ds_teste.copy()


    # Input Treino
    X_train = ds[MODELO_ATUAL]
    imputer_linear = IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42)
    X_train_imputado = imputer_linear.fit_transform(X_train)

    # Input Teste
    X_test= dst[MODELO_ATUAL]
    imputer_nao_linear = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=100, random_state=42),
    max_iter=10,
    random_state=42
    )
    X_imputado = inputer_nao_linear.fit_transform(X_test)


    #Treinamento
    


    #Avaliação



    #Retorno


def MiceNonLinear(ds_treino,ds_teste):
    ds = ds_treino.copy()
    dst = ds_teste.copy()


    # Input Treino
    X_train = ds[MODELO_ATUAL]
    imputer_linear = IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42)
    X_train_imputado = imputer_linear.fit_transform(X_train)

    # Input Teste
    X_test= dst[MODELO_ATUAL]
    imputer_nao_linear = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=100, random_state=42),
    max_iter=10,
    random_state=42
    )
    X_imputado = inputer_nao_linear.fit_transform(X_test)


    #Treinamento

    

    #Avaliação



    #Retorno



dx_treino = dx_treino.copy()
X = dx_treino[MODELO_ATUAL]

X

X

,306-AIT-004-2,306-AIT-004-5,306FE002-AI001-R,406-AI-004-2,406-AI-004-4,406-AIT-001,Hora,Minuto,Dia_Semana,score_atual,score_max_30
0,71.445,183.992,7.277,43.072,178.567,1.008,14,15,3,1.0,1.0
1,67.433,186.557,7.291,45.751,178.031,1.036,14,20,3,1.0,1.0
2,66.290,181.367,6.485,41.679,174.437,0.919,14,25,3,1.0,1.0
3,68.476,174.032,8.987,40.868,176.775,0.555,14,30,3,1.0,1.0
4,67.240,173.400,7.643,42.162,176.990,0.852,14,35,3,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
11593,62.172,239.748,19.285,127.243,258.270,0.787,22,50,4,0.0,0.0
11594,61.412,237.278,7.195,127.085,268.000,0.485,22,55,4,0.0,0.0
11595,60.070,239.702,5.026,131.606,282.637,0.514,23,0,4,0.0,0.0
11596,60.468,232.897,4.778,132.394,272.372,0.807,23,5,4,0.0,0.0
